# Importing Required Libraries

In [70]:
from google import genai
import os
from dotenv import load_dotenv

# Load and Retrieve API Key from .env

In [71]:
# Charger les variables du fichier .env
load_dotenv()

# Récupérer la clé
api_key = os.getenv("GEMINI_API_KEY")

# Call Gemini API 

In [72]:
client = genai.Client(api_key=api_key)


# Initialize Chat Session with Gemini Model

In [107]:
session = client.chats.create(model="gemini-2.0-flash")#gemini-2.5-flash

# Define Prompt for Bio-to-JSON Extraction

In [110]:
prompt = """
You are an AI system that extracts structured information from short professional bios into valid JSON. 
Return ONLY a valid JSON object with exactly these fields, in this order:  
- current_company (string)  
- past_companies (array of strings)  
- topics_of_interest (array of strings)  
- speaking_experience (array of strings)  

 Important Rules:
- Only extract fields: current_company, past_companies, topics_of_interest, speaking_experience
- If a field is missing, return [] or ""
- Only extract information explicitly mentioned in the bio. Do NOT guess or infer.
- Output must be valid JSON: without markdown,without  ```json,without  quotes around the JSON, and without  \\n characters
-Format the output like (no extra newlines unless inside array items):
{
  "current_company":"Company",
  "past_companies":["Past1"],
 "topics_of_interest":["Topic1"],
"speaking_experience":["Event1","Event2"]
}


Example 1:
Bio: "Software engineer at Microsoft. Ex-IBM. Interested in cloud computing and AI. Speaker at DevCon."
Output:
{
  "current_company": "Microsoft",
  "past_companies": ["IBM"],
  "topics_of_interest": ["cloud computing", "AI"],
  "speaking_experience": ["DevCon"]
}

Example 2:
Bio: "Marketing lead at HubSpot. Ex-Salesforce. Passionate about growth hacking and digital marketing."
Output:
{
  "current_company": "HubSpot",
  "past_companies": ["Salesforce"],
  "topics_of_interest": ["growth hacking", "digital marketing"],
  "speaking_experience": []
}

Example 3:
Bio: "Product manager at Spotify. Ex-Snapchat. Loves music tech and mobile apps. Speaker at Web Summit and TechCrunch."
Output:
{
  "current_company": "Spotify",
  "past_companies": ["Snapchat"],
  "topics_of_interest": ["music tech", "mobile apps"],
  "speaking_experience": ["Web Summit", "TechCrunch"]
}
"""

# Run Prompt

In [111]:
# Send the system message as the first message in the session
session.send_message(prompt);


# Define Bio Extraction Helper Function

In [82]:
# ----------------------------
def extract_bio_info_session(bio_text, session):
    # Send bio as user message
    response = session.send_message(f"Now extract info from the following bio. Remember: do NOT infer or guess anything beyond the bio text:\n {bio_text}")
    llm_output = response.text

    return llm_output 


# Test 

In [116]:
bio1 = "Currently building GTM at Fireblocks. Ex-Coinbase. Passionate about fintech growth. Speaker at EthCC and Web Summit."
bio2 = "Senior developer at OpenAI. Ex-Google. Interested in AI research and robotics. Speaker at NeurIPS."
bio3="as a senior developer. Previously at Google. Robotics and AI research excite me. Speaker at NeurIPS."

In [118]:
print(extract_bio_info_session(bio1, session))

{
  "current_company": "Fireblocks",
  "past_companies": ["Coinbase"],
  "topics_of_interest": ["fintech growth"],
  "speaking_experience": ["EthCC", "Web Summit"]
}

